In [30]:
import os
os.chdir('/content/drive/MyDrive/credit-risk-assessment-system')

import pandas as pd
import numpy as np
import joblib

!pip install mlflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 77.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [32]:
import mlflow

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('credit-risk-assessment')

print("MLflow tracking set up")

2026/08/08 13:13:14 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/08 13:13:14 INFO mlflow.store.db.utils: Updating database tables
2026/08/08 13:13:19 INFO mlflow.tracking.fluent: Experiment with name 'credit-risk-assessment' does not exist. Creating a new experiment.


MLflow tracking set up


In [33]:
gitignore_path = '.gitignore'
with open(gitignore_path, 'a') as f:
    f.write('\nmlflow.db\n')
print("Updated .gitignore")

Updated .gitignore


In [34]:
runs_to_log = [
    {
        'run_name': 'logistic_regression_baseline',
        'params': {'model_type': 'LogisticRegression', 'max_iter': 1000},
        'metrics': {'auc_roc': 0.7410}
    },
    {
        'run_name': 'decision_tree_baseline',
        'params': {'model_type': 'DecisionTree', 'max_depth': 8},
        'metrics': {'auc_roc': 0.7186}
    },
    {
        'run_name': 'xgboost_default',
        'params': {'model_type': 'XGBoost', 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1},
        'metrics': {'auc_roc': 0.7615, 'cv_mean_auc': 0.7555, 'cv_std': 0.001538}
    },
    {
        'run_name': 'lightgbm_default',
        'params': {'model_type': 'LightGBM', 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1},
        'metrics': {'auc_roc': 0.7606, 'cv_mean_auc': 0.7557, 'cv_std': 0.001089}
    },
    {
        'run_name': 'catboost_default',
        'params': {'model_type': 'CatBoost', 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1},
        'metrics': {'auc_roc': 0.7611, 'cv_mean_auc': 0.7570, 'cv_std': 0.001512}
    },
    {
        'run_name': 'catboost_tuned_optuna',
        'params': {'model_type': 'CatBoost', 'iterations': 477, 'depth': 7,
                    'learning_rate': 0.042106, 'l2_leaf_reg': 7.262614, 'tuning_method': 'Optuna_30trials'},
        'metrics': {'cv_mean_auc': 0.7583, 'cv_std': 0.0016, 'test_auc': 0.7620}
    },
    {
        'run_name': 'catboost_final_fair',
        'params': {'model_type': 'CatBoost', 'iterations': 477, 'depth': 7,
                    'learning_rate': 0.042106, 'l2_leaf_reg': 7.262614,
                    'class_weighting': 'Balanced', 'gender_excluded': True},
        'metrics': {'test_auc': 0.7604, 'optimal_threshold': 0.7199,
                     'precision_at_threshold': 0.2961, 'recall_at_threshold': 0.2961}
    }
]

for run_config in runs_to_log:
    with mlflow.start_run(run_name=run_config['run_name']):
        mlflow.log_params(run_config['params'])
        mlflow.log_metrics(run_config['metrics'])

print(f"Logged {len(runs_to_log)} runs to MLflow")

Logged 7 runs to MLflow


In [35]:
with mlflow.start_run(run_name='catboost_final_fair_with_artifact'):
    mlflow.log_params({
        'model_type': 'CatBoost', 'iterations': 477, 'depth': 7,
        'learning_rate': 0.042106, 'gender_excluded': True
    })
    mlflow.log_metrics({'test_auc': 0.7604})
    mlflow.catboost.log_model(catboost_fair, 'model')

print("Final model artifact logged")

2026/08/08 13:22:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Final model artifact logged
